# Демо проектных упражнений модуля 7 — Балансировка, совместные назначения и вмешательства

**Демо-реализация, не готовое решение полномасштабного проекта.** Синтетические кластеры позволяют проследить маршруты. Роутер и эксперты здесь не обучаются, а кластеры не являются доказанными семантическими специализациями.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'

plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":12, "axes.titlesize":15, "axes.labelsize":13, "xtick.labelsize":11, "ytick.labelsize":11, "figure.dpi":150})


## Демо 1. Балансировка при неизменных оценках

Создадим 900 векторов из трёх синтетических кластеров и шесть экспертов с выбором top-2. Сам роутер не обучается: сто раз меняются только смещения выбора. На каждом шаге сравниваем нагрузки с их средним и делаем шаг по знаку отклонения. Это позволяет увидеть работу контроллера отдельно от обучения весов.

Слева проследите коэффициент вариации нагрузок. Справа каждая колонка делится на 600 — число назначений для 300 токенов кластера при top-2. Поэтому колонка суммируется к единице и показывает долю назначений, а не долю токенов, для которых эксперт выбран хотя бы раз.

In [ ]:
ru = DEMO_LANGUAGE == "ru"
rng = np.random.default_rng(SEED)
cluster = np.repeat(np.arange(3), 300)
features = rng.normal(size=(900, 8)) + np.eye(3, 8)[cluster] * 2
router_w = rng.normal(size=(8, 6))
logits = features @ router_w
log_scores = -np.logaddexp(0., -logits)
scores = np.exp(log_scores)

def route(bias):
    indices = np.argsort(-(scores + bias), axis=1, kind="stable")[:, :2]
    selected = np.take_along_axis(log_scores, indices, axis=1)
    gates = np.exp(selected - selected.max(axis=1, keepdims=True))
    gates /= gates.sum(axis=1, keepdims=True)
    return indices, gates, np.bincount(indices.ravel(), minlength=6)

bias = np.zeros(6)
history = []
for step in range(101):
    top2, gates, counts = route(bias)
    history.append(counts.std() / counts.mean())
    if step < 100:
        bias += .01 * np.sign(counts.mean() - counts)
assert history[-1] < history[0]
assert counts.sum() == 1800
print("CV:", round(history[0], 4), "→", round(history[-1], 4))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained")
axes[0].plot(history, color="#5E81AC")
axes[0].set(xlabel="Обновления смещений" if ru else "Bias updates", ylabel="CV",
            title="Фиксированные оценки" if ru else "Fixed router scores")
heat = np.zeros((6, 3), dtype=int)
for c, selected in zip(cluster, top2):
    heat[selected, c] += 1
im = axes[1].imshow(heat / 600, cmap="Blues", vmin=0, vmax=1, aspect="auto")
axes[1].set(xlabel="Синтетический кластер" if ru else "Synthetic cluster",
            ylabel="Эксперт" if ru else "Expert", xticks=[0,1,2],
            title="Доля назначений в кластере" if ru else "Assignment share within cluster")
fig.colorbar(im, ax=axes[1])
assert np.allclose((heat / 600).sum(axis=0), 1.)
plt.show()

Снижение CV показывает более равномерную нагрузку в этой популяции оценок. Различия между колонками могут сохраниться: общий баланс не требует одинаковых маршрутов каждого кластера. Ни качество задачи, ни семантические навыки здесь не измеряются.

## Демо 2. Частота совместного выбора и эффект вмешательства

Для каждой пары top-2 запишем два направленных счётчика. После деления строки на частоту выбора эксперта получим распределение его партнёра. Затем зададим фиксированные случайные FFN и по очереди обнулим вклад каждого эксперта. Маршруты и остальные веса смеси оставим прежними: перенормировка или повторный выбор были бы другими вмешательствами.

In [ ]:
# Count co-activation separately from the effect of changing expert outputs.
co = np.zeros((6, 6), dtype=int)
for a, b in top2:
    co[a, b] += 1
    co[b, a] += 1
assert np.all(co == co.T) and np.all(np.diag(co) == 0)
assert np.array_equal(co.sum(axis=1), counts)  # exactly one partner at top-2
conditional = co / counts[:, None]
assert np.allclose(conditional.sum(axis=1), 1.)
w1 = rng.normal(scale=.3, size=(6, 8, 12))
w2 = rng.normal(scale=.3, size=(6, 12, 8))
expert_outputs = np.stack([np.tanh(features @ a) @ b for a,b in zip(w1,w2)], axis=1)
selected_outputs = expert_outputs[np.arange(len(features))[:, None], top2]
original = (gates[..., None] * selected_outputs).sum(axis=1)
effects = []
for expert in range(6):
    # Zero an expert contribution while keeping routes and all other gates fixed.
    kept = gates * (top2 != expert)
    intervened = (kept[..., None] * selected_outputs).sum(axis=1)
    effects.append(np.mean((intervened - original)**2))
assert np.all(np.isfinite(effects)) and min(effects) >= 0
fig, axes = plt.subplots(1,2,figsize=(11,4.8),layout="constrained")
im=axes[0].imshow(conditional,cmap="Blues",vmin=0,vmax=1)
axes[0].set(xlabel="Партнёр" if ru else "Partner expert",ylabel="Выбранный эксперт" if ru else "Selected expert",
    title="P(партнёр | эксперт)" if ru else "P(partner | expert)")
fig.colorbar(im,ax=axes[0])
axes[1].bar(range(6),effects,color="#D08770")
axes[1].set(xlabel="Обнулённый эксперт" if ru else "Zeroed expert",
    ylabel="MSE выхода" if ru else "Output MSE",
    title="При неизменных маршрутах" if ru else "With routes held fixed")
plt.show()
print("Это изменение вектора, не падение качества задачи." if ru else "This measures vector change, not task-quality loss.")

Матрица совместного выбора до нормировки симметрична, но условные вероятности не обязаны быть симметричными: знаменатели строк различаются. Столбцы второго графика измеряют средний квадрат изменения выходного вектора. Чтобы говорить о потере способности, понадобится обученная модель и метрика соответствующей задачи.

## Переход к полному проекту

На обученной модели собирайте маршруты на отложенных документах. Сравните частоты, условные совместные назначения и изменение метрики задачи при объявленной политике вмешательства. Отделяйте обнуление вклада от перенормировки и повторной маршрутизации.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.